In [1]:
import os
import sqlite3
from time import perf_counter
from dotenv import load_dotenv
 
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import MessagesState, START, StateGraph, END
from langgraph.prebuilt import ToolNode
from langsmith import Client, evaluate

 
load_dotenv()
 
required_keys = ["OPENAI_API_KEY", "TAVILY_API_KEY"]
missing_keys = [key for key in required_keys if not os.environ.get(key)]
 
if missing_keys:
    raise ValueError(f"Missing the following keys from the .env file: {missing_keys}")
 
if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "exercise-4-agent-with-tools"

C:\Users\User\anaconda3\envs\ai-project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b.
 
    Args:
        a: first int
        b: second int
    """
    return a * b
 
 
@tool
def add(a: int, b: int) -> int:
    """Add a and b.
 
    Args:
        a: first int
        b: second int
    """
    return a + b
 
 
@tool
def divide(a: int, b: int) -> float:
    """Divide a by b.
 
    Args:
        a: first int
        b: second int
    """
    return a / b

In [3]:
@tool
def search_web(query: str) -> str:
    """Search the web for current information using Tavily.
 
    Args:
        query: question or search query
    """
 
    try:
        import json
        import subprocess
 
        payload = json.dumps({
            "api_key": os.environ["TAVILY_API_KEY"],
            "query": query,
            "max_results": 3,
            "search_depth": "advanced",
        })
 
        result = subprocess.run(
            ["curl", "-s", "https://api.tavily.com/search",
             "-X", "POST",
             "-H", "Content-Type: application/json",
             "-d", payload],
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=30,
        )
 
        if result.returncode != 0:
            return f"Tavily search failed: curl error: {result.stderr}"
 
        response = json.loads(result.stdout)
        search_docs = response.get("results", [])
    except Exception as error:
        return f"Tavily search failed: {error}"
 
    if not search_docs:
        return "No useful Tavily search results were found."
 
    formatted_search_docs = "\n\n---\n\n".join(
        (f'<Document href="{doc.get("url", "")}">\n'
            f'Title: {doc.get("title", "")}\n'
            f'{doc.get("content", "")}\n'
            f"</Document>") for doc in search_docs)
 
    return formatted_search_docs

In [4]:
conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.execute("""CREATE TABLE todos (id INTEGER PRIMARY KEY,
                                    task TEXT NOT NULL,
                                    deadline TEXT)""")
 
conn.executemany("""INSERT INTO todos (task, deadline) VALUES (?, ?)""",
                 [("Finish booking travel to Hong Kong",
                   "End of next week"),
                  ("Call parents back about Thanksgiving plans",
                   None),
                  ("Drop by Yoga in person",
                   "Sunday")])
conn.commit()
 
 
@tool
def database_lookup(query: str) -> str:
    """Execute one CRUD SQL statement against the SQLite database.
 
    Supported operations: SELECT, INSERT, UPDATE, and DELETE.
 
    Args:
        query: one complete SQLite statement
    """
 
    query = query.strip()
 
    if not query:
        return "SQL_ERROR: The query cannot be empty."
 
    operation = query.split(maxsplit=1)[0].upper()
    allowed_operations = {"SELECT", "INSERT", "UPDATE", "DELETE"}
 
    if operation not in allowed_operations:
        return ("SQL_ERROR: Only SELECT, INSERT, UPDATE, "
                "and DELETE statements are allowed.")
 
    try:
        cursor = conn.execute(query)
 
        if operation == "SELECT":
            rows = cursor.fetchall()
            columns = [description[0] for description in cursor.description]
 
            if not rows:
                return "The query returned no records."
 
            formatted_rows = []
            for row in rows:
                values = [f"{column}: {value}" for column, value in zip(columns, row)]
                formatted_rows.append("- " + ", ".join(values))
            return "\n".join(formatted_rows)
 
        conn.commit()
        return f"{operation} completed successfully. Rows affected: {cursor.rowcount}"
 
    except sqlite3.Error as error:
        conn.rollback()
        return f"SQL_ERROR: {error}"

In [5]:
tools = [add, multiply, divide, search_web, database_lookup]
 
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_retries=3)
 
llm_with_tools = llm.bind_tools(tools, tool_choice="auto", parallel_tool_calls=False)

In [6]:
sys_msg = SystemMessage(
    content=("You are a helpful assistant with access to exactly these tools: "
             "add, multiply, divide, search_web, and database_lookup. "

             "The SQLite database has this schema:\n"
             "todos(\n"
             "    id INTEGER PRIMARY KEY,\n"
             "    task TEXT NOT NULL,\n"
             "    deadline TEXT\n"
             ")\n"

             "Use database_lookup whenever the user wants to read or modify "
             "database data. Translate the user's request into one valid "
             "SQLite SELECT, INSERT, UPDATE, or DELETE statement. "
             "Never invent tables or columns. "
             "Before UPDATE or DELETE, ensure that the WHERE condition "
             "matches the user's requested scope. "
             "Use SELECT when the user asks to view, list, find, count, "
             "filter, or summarize database records. "

             "Select the appropriate tool for the user's request. "
             "Call only one tool at a time. After receiving a tool result, "
             "decide whether another available tool is necessary. "
             "After search_web returns results, summarize those results directly. "
             "Do not attempt to open URLs or call open_url, browser, fetch, or any "
             "tool that is not explicitly available. Never invent tool names. "
             "When all necessary tools have been used, provide the final answer."))
 
 
def assistant(state: MessagesState):
    """Select the appropriate tool or produce the final answer."""
 
    response = llm_with_tools.invoke([sys_msg] + state["messages"])
    return {"messages": [response]}

In [7]:
def route_tool_call(state: MessagesState) -> str:
    """Route to the specific tool requested by the assistant."""
 
    last_message = state["messages"][-1]
 
    if not last_message.tool_calls:
        return "end"
 
    tool_name = last_message.tool_calls[0]["name"]
 
    routes = {"add": "add_tool",
              "multiply": "multiply_tool",
              "divide": "divide_tool",
              "search_web": "search_web_tool",
              "database_lookup": "database_lookup_tool"}
 
    return routes.get(tool_name, "end")
 
 
builder = StateGraph(MessagesState)
 
builder.add_node("assistant", assistant)
builder.add_node("add_tool", ToolNode([add]))
builder.add_node("multiply_tool", ToolNode([multiply]))
builder.add_node("divide_tool", ToolNode([divide]))
builder.add_node("search_web_tool", ToolNode([search_web]))
builder.add_node("database_lookup_tool", ToolNode([database_lookup]))
 
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", route_tool_call, {
        "add_tool": "add_tool",
        "multiply_tool": "multiply_tool",
        "divide_tool": "divide_tool",
        "search_web_tool": "search_web_tool",
        "database_lookup_tool": "database_lookup_tool",
        "end": END})
 
for tool_node in ["add_tool", "multiply_tool", "divide_tool",
                  "search_web_tool", "database_lookup_tool"]:
    builder.add_edge(tool_node, "assistant")
 
graph = builder.compile()

In [8]:
if __name__ == "__main__":
    search_test = graph.invoke(
        {"messages": [HumanMessage(content=(
            "Use the web-search tool to find the latest "
            "information about LangGraph."))]},
        config={"configurable": {"thread_id": "search-test-1"}},
    )
    search_test["messages"][-1].pretty_print()

    calculator_test = graph.invoke(
        {"messages": [HumanMessage(content="Use the multiplication tool to multiply 8 by 7.")]},
        config={"configurable": {"thread_id": "calculator-test-1"}},
    )
    calculator_test["messages"][-1].pretty_print()

    database_test = graph.invoke(
        {"messages": [HumanMessage(content=(
            "Use the database lookup tool to find the task about Hong Kong."))]},
        config={"configurable": {"thread_id": "database-test-1"}},
    )
    database_test["messages"][-1].pretty_print()

    mixed_test = graph.invoke(
        {"messages": [HumanMessage(content=(
            "First, use the database lookup tool to find the Yoga task. "
            "Then use the multiplication tool to multiply 3 by 3."))]},
        config={"configurable": {"thread_id": "mixed-test-1"}},
    )
    mixed_test["messages"][-1].pretty_print()

    # ---- 8.2 LangSmith evaluation dataset ----

    if not os.environ.get("LANGSMITH_API_KEY"):
        raise ValueError(
            "LANGSMITH_API_KEY must also be stored in the .env file "
            "to create and evaluate the LangSmith dataset.")

    client = Client()
    dataset_name = "exercise-4-agent-tool-selection"

    existing_datasets = list(client.list_datasets(dataset_name=dataset_name))

    if existing_datasets:
        dataset = existing_datasets[0]
    else:
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description=("Tests calculator, web-search, database-lookup, "
                         "and mixed-tool selection."))

        evaluation_inputs = [
            {"question": "Multiply 6 by 9."},
            {"question": "Search the web for the latest information about LangGraph."},
            {"question": "Look in the ToDo database for the task about Hong Kong."},
            {"question": "Look up the Yoga task in the database, and multiply 3 by 3."},
        ]

        evaluation_outputs = [
            {"expected_tools": ["multiply"]},
            {"expected_tools": ["search_web"]},
            {"expected_tools": ["database_lookup"]},
            {"expected_tools": ["database_lookup", "multiply"]},
        ]

        client.create_examples(
            inputs=evaluation_inputs,
            outputs=evaluation_outputs,
            dataset_id=dataset.id,
        )

    def run_agent(inputs: dict) -> dict:
        """Run the agent and return its answer, selected tools, and timing."""

        evaluation_thread = {"configurable": {"thread_id": f"evaluation-{inputs['question']}"}}

        start_time = perf_counter()
        result = graph.invoke(
            {"messages": [HumanMessage(content=inputs["question"])]},
            config=evaluation_thread,
        )
        total_time = perf_counter() - start_time

        selected_tools = []
        for message in result["messages"]:
            tool_calls = getattr(message, "tool_calls", [])
            for tool_call in tool_calls:
                selected_tools.append(tool_call["name"])

        return {
            "answer": result["messages"][-1].content,
            "selected_tools": selected_tools,
            "total_time_seconds": round(total_time, 4),
        }

    def correct_tool_selection(outputs: dict, reference_outputs: dict) -> bool:
        """Check whether the agent selected every expected tool."""

        selected_tools = set(outputs.get("selected_tools", []))
        expected_tools = set(reference_outputs.get("expected_tools", []))

        return expected_tools.issubset(selected_tools)

    def no_unexpected_tools_used(outputs: dict, reference_outputs: dict) -> bool:
        """Check that the agent didn't call tools outside the expected set."""

        selected_tools = set(outputs.get("selected_tools", []))
        expected_tools = set(reference_outputs.get("expected_tools", []))

        return selected_tools.issubset(expected_tools)

    def answer_present(outputs: dict) -> bool:
        """Check that the agent returned a non-empty final answer."""

        answer = outputs.get("answer")
        return bool(isinstance(answer, str) and answer.strip())

    def acceptable_latency(outputs: dict) -> bool:
        """
        Example latency threshold.

        Change 15 seconds if a different limit is appropriate for
        your machine or internet connection.
        """
        return outputs.get("total_time_seconds", float("inf")) <= 15

    evaluation_results = evaluate(
        run_agent,
        data=dataset_name,
        evaluators=[correct_tool_selection,
                    no_unexpected_tools_used,
                    answer_present,
                    acceptable_latency],
        experiment_prefix="exercise-4-tool-agent",
    )

================================== Ai Message ==================================

Here are the latest updates on LangGraph:

1. **Version Updates**: LangGraph has recently released versions 1.1 and 1.2, with the latest tagged release being 1.2.6 on June 18, 2026. The framework has evolved significantly since its v1.0 milestone in October 2025, which was backed by $125 million in Series B funding.

2. **New Features**: The updates include new features such as model profiles that provide detailed information about the models used, including input and output token limits, tool calling support, and structured output capabilities. These enhancements are aimed at improving the development experience for building AI agents.

3. **Deployment Options**: LangGraph now offers a platform for managed deployment, allowing developers to expose their agents as production REST APIs without the need to manage infrastructure. This includes support for both self-hosted and cloud-hosted deployment options.

4it [00:15,  3.91s/it]
